# Structured Unit Sum timm Benchmark

This notebook displays the latest benchmark results written by `benchmark_timm_structured_unit_sum.py`. The benchmark compares the optimized `StructuredUnitSum` implementation with an allocation-heavy naive runner using the same reducer plan.

## Latest Captured Result

- Model: `vit_tiny_patch16_224`, `img_size=32`, `batch_size=1`, CPU
- Plan: 25 dependency groups, 122 reducer mappings, output length 16,320
- Correctness: max absolute difference `0.0`
- `StructuredUnitSum`: median `4.1549 ms` per call
- Naive runner: median `6.5088 ms` per call
- Speedup: `1.57x`, saving `2.3539 ms` per call

These values come from `results/latest.json`, generated on 2026-05-07 with PyTorch 2.9.0 and timm 1.0.25.

In [ ]:
from pathlib import Path
import json

benchmark_dir = Path.cwd()
if benchmark_dir.name != "structured_unit_sum":
    benchmark_dir = Path("benchmarks/structured_unit_sum")

results_path = benchmark_dir / "results" / "latest.json"
print(f"Loading {results_path}")

with results_path.open(encoding="utf-8") as f:
    result = json.load(f)

result["created_at"], result["environment"], result["model"]

In [ ]:
rows = [
    {
        "implementation": "StructuredUnitSum",
        **{k: result["timing"]["structured"][k] for k in ["median_ms", "mean_ms", "min_ms", "max_ms", "stdev_ms"]},
    },
    {
        "implementation": "Naive",
        **{k: result["timing"]["naive"][k] for k in ["median_ms", "mean_ms", "min_ms", "max_ms", "stdev_ms"]},
    },
]

try:
    import pandas as pd
    display(pd.DataFrame(rows).style.format({
        "median_ms": "{:.4f}",
        "mean_ms": "{:.4f}",
        "min_ms": "{:.4f}",
        "max_ms": "{:.4f}",
        "stdev_ms": "{:.4f}",
    }))
except ImportError:
    for row in rows:
        print(row)

print(f"Speedup: {result['timing']['speedup_x']:.2f}x")
print(f"Median time saved: {result['timing']['median_ms_saved']:.4f} ms per call")
print(f"Max absolute difference: {result['correctness']['max_abs_diff']:.3g}")

In [ ]:
try:
    import matplotlib.pyplot as plt

    labels = ["StructuredUnitSum", "Naive"]
    medians = [
        result["timing"]["structured"]["median_ms"],
        result["timing"]["naive"]["median_ms"],
    ]
    colors = ["#2563eb", "#9ca3af"]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(labels, medians, color=colors)
    ax.set_ylabel("Median time per call (ms)")
    ax.set_title("Structured unit sum speed")
    ax.bar_label(bars, fmt="%.4f ms")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()
except ImportError:
    print("matplotlib is not installed; table output above is still available.")

In [ ]:
# Re-run the default benchmark from this notebook if desired.
# import subprocess, sys
# script = benchmark_dir / "benchmark_timm_structured_unit_sum.py"
# subprocess.run([
#     sys.executable,
#     str(script),
#     "--model", "vit_tiny_patch16_224",
#     "--img-size", "32",
#     "--iterations", "200",
#     "--repeats", "7",
# ], check=True)